# Classificação de Variedades de Grãos de Trigo — Seeds Dataset

**Metodologia:** CRISP-DM  
**Dataset:** Seeds Dataset — UCI Machine Learning Repository  

| | |
|---|---|
| **Aluno** | Lucas Michels Kuntz |
| **Disciplina** | Fase 4 — Cap 03: Implementando Algoritmos de Machine Learning com Scikit-learn |
| **Data** | Junho/2026 |

---

## Sumário
1. [Entendimento do Negócio](#1)
2. [Análise Exploratória e Pré-processamento](#2)
3. [Implementação e Comparação de Classificadores](#3)
4. [Otimização de Hiperparâmetros](#4)
5. [Interpretação dos Resultados](#5)

<a id='1'></a>
## 1. Entendimento do Negócio (CRISP-DM — Fase 1)

Em cooperativas agrícolas de pequeno porte, a classificação de grãos de trigo é realizada **manualmente por especialistas** — um processo lento e sujeito a erros humanos. Automatizar essa triagem com aprendizado de máquina pode reduzir custos operacionais, aumentar a precisão e acelerar o fluxo de produção.

### Objetivo de Negócio
Construir um modelo capaz de **classificar automaticamente** amostras de grão de trigo em uma das três variedades:
- **Kama** (classe 1)
- **Rosa** (classe 2)  
- **Canadian** (classe 3)

usando apenas características físicas mensuráveis por sensores ópticos.

### Critério de Sucesso
Acurácia ≥ 90% no conjunto de teste, com F1-score equilibrado entre as três classes (sem viés para uma variedade específica).

### Atributos do Dataset
| # | Atributo | Descrição |
|---|---|---|
| 1 | Área | Medida da área da seção transversal do grão |
| 2 | Perímetro | Comprimento do contorno do grão |
| 3 | Compacidade | Calculada como $4\pi \cdot \text{Área} / \text{Perímetro}^2$ |
| 4 | Comprimento do Núcleo | Comprimento do eixo principal da elipse equivalente |
| 5 | Largura do Núcleo | Comprimento do eixo secundário da elipse |
| 6 | Coeficiente de Assimetria | Medida de assimetria do grão |
| 7 | Comprimento do Sulco | Comprimento do sulco central do grão |

In [ ]:
# ── Instalação de dependências (executar apenas se necessário) ──────────────
# !pip install pandas numpy matplotlib seaborn scikit-learn ucimlrepo

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

# Scikit-learn — pré-processamento
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Scikit-learn — modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

# Scikit-learn — métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Estilo global
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})
SEED = 42
CLASS_NAMES = ['Kama', 'Rosa', 'Canadian']
PALETTE = ['#2196F3', '#E91E63', '#4CAF50']

print('Bibliotecas carregadas com sucesso.')

<a id='2'></a>
## 2. Entendimento e Pré-processamento dos Dados (CRISP-DM — Fases 2 e 3)

### 2.1 Carregamento do Dataset

O Seeds Dataset está disponível no UCI ML Repository. A célula abaixo tenta carregá-lo via `ucimlrepo`; caso não esteja instalado, faz o download direto da URL pública.

In [ ]:
COLS = [
    'area', 'perimetro', 'compacidade',
    'comp_nucleo', 'larg_nucleo',
    'assimetria', 'comp_sulco', 'variedade'
]

try:
    from ucimlrepo import fetch_ucirepo
    seeds = fetch_ucirepo(id=236)
    X_raw = seeds.data.features
    y_raw = seeds.data.targets
    df = pd.concat([X_raw, y_raw], axis=1)
    df.columns = COLS
    print('Dataset carregado via ucimlrepo.')
except Exception:
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt'
    df = pd.read_csv(url, sep=r'\s+', header=None, names=COLS)
    print('Dataset carregado via download direto.')

# Mapeia classes numéricas para nomes
df['variedade'] = df['variedade'].map({1: 'Kama', 2: 'Rosa', 3: 'Canadian'})

print(f'\nShape: {df.shape[0]} amostras × {df.shape[1]} colunas')
df.head(10)

### 2.2 Análise Exploratória Inicial (EDA)

In [ ]:
print('=== Tipos de Dados e Nulos ===' )
df.info()
print()
print('=== Valores Ausentes ===')
print(df.isnull().sum())
print()
print('=== Duplicatas ===')
print(f'{df.duplicated().sum()} linha(s) duplicada(s)')

**Observação:** O dataset está completamente limpo — sem valores ausentes e sem duplicatas. Não há necessidade de imputação ou remoção de registros.

In [ ]:
print('=== Distribuição das Classes ===')
print(df['variedade'].value_counts())
print()
print('Proporção:')
print(df['variedade'].value_counts(normalize=True).map('{:.1%}'.format))

**Observação:** O dataset é **perfeitamente balanceado** — 70 amostras por classe (33,3% cada). Não há necessidade de técnicas de resampling (SMOTE, undersampling etc.).

### 2.3 Estatísticas Descritivas

In [ ]:
FEATURES = ['area', 'perimetro', 'compacidade', 'comp_nucleo',
            'larg_nucleo', 'assimetria', 'comp_sulco']

desc = df[FEATURES].agg(['mean', 'median', 'std', 'min', 'max']).T
desc.columns = ['Média', 'Mediana', 'Desvio Padrão', 'Mínimo', 'Máximo']
desc.index.name = 'Atributo'
desc.round(4)

In [ ]:
print('=== Estatísticas Descritivas por Variedade ===')
df.groupby('variedade')[FEATURES].mean().round(3)

**Insights das estatísticas por variedade:**
- **Rosa** apresenta os maiores valores de área (~18,3), perímetro (~16,1) e comprimentos — é a variedade com grãos de maior porte.
- **Canadian** tem os menores valores em quase todos os atributos de tamanho — grão mais compacto e pequeno.
- **Kama** ocupa posição intermediária.
- O **coeficiente de assimetria** é a feature com maior variação relativa entre classes: Canadian (~1.8) vs Rosa (~3.7).

### 2.4 Visualizações

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

feat_labels = {
    'area': 'Área',
    'perimetro': 'Perímetro',
    'compacidade': 'Compacidade',
    'comp_nucleo': 'Comp. Núcleo',
    'larg_nucleo': 'Larg. Núcleo',
    'assimetria': 'Assimetria',
    'comp_sulco': 'Comp. Sulco'
}

for i, (feat, label) in enumerate(feat_labels.items()):
    ax = axes[i]
    for j, (var, color) in enumerate(zip(CLASS_NAMES, PALETTE)):
        subset = df.loc[df['variedade'] == var, feat]
        ax.hist(subset, bins=15, alpha=0.55, color=color, label=var, edgecolor='white')
    ax.axvline(df[feat].mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'Média={df[feat].mean():.2f}')
    ax.set_title(label)
    ax.legend(fontsize=7)

axes[-1].set_visible(False)
fig.suptitle('Histogramas por Atributo — Distribuição por Variedade', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('seeds_histogramas.png', bbox_inches='tight')
plt.show()

**Interpretação dos Histogramas:**
- **Área, Perímetro, Comprimento e Largura do Núcleo:** as três variedades mostram distribuições deslocadas umas das outras, indicando alto poder discriminativo. Rosa (rosa/magenta) concentra-se em valores maiores; Canadian (verde) em valores menores.
- **Compacidade:** as distribuições se sobrepõem mais, com Kama e Canadian mais próximos. Menor poder discriminativo isolado.
- **Assimetria:** Canadian apresenta valores visivelmente menores (~1–3), enquanto Rosa e Kama se distribuem mais amplamente. Boa feature para separar Canadian.
- **Comprimento do Sulco:** padrão similar ao comprimento do núcleo — Rosa em valores mais altos, Canadian em mais baixos.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (feat, label) in enumerate(feat_labels.items()):
    ax = axes[i]
    data_by_class = [df.loc[df['variedade'] == var, feat].values for var in CLASS_NAMES]
    bp = ax.boxplot(
        data_by_class, labels=CLASS_NAMES, patch_artist=True,
        medianprops=dict(color='black', linewidth=2)
    )
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(label)
    ax.tick_params(axis='x', labelsize=9)

axes[-1].set_visible(False)
fig.suptitle('Box Plots por Atributo — Comparação entre Variedades', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('seeds_boxplots.png', bbox_inches='tight')
plt.show()

**Interpretação dos Box Plots:**
- **Área e Perímetro:** as três caixas são claramente separadas, com mediana Rosa > Kama > Canadian e sem sobreposição nas medianas. Excelentes discriminadores.
- **Compacidade:** Kama possui mediana ligeiramente maior e Canadian ligeiramente menor; boxes parcialmente sobrepostos — menos discriminativo.
- **Assimetria:** Canadian apresenta caixa estreita e baixa (~1,5–3), enquanto Kama e Rosa têm maior variância e valores mais altos. Útil para identificar Canadian.
- Os poucos outliers visíveis não comprometem a qualidade do dataset.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scatter_pairs = [
    ('area', 'perimetro', 'Área × Perímetro'),
    ('comp_nucleo', 'larg_nucleo', 'Comp. Núcleo × Larg. Núcleo'),
    ('assimetria', 'compacidade', 'Assimetria × Compacidade'),
]

for ax, (fx, fy, title) in zip(axes, scatter_pairs):
    for var, color in zip(CLASS_NAMES, PALETTE):
        sub = df[df['variedade'] == var]
        ax.scatter(sub[fx], sub[fy], c=color, label=var, alpha=0.7, s=45, edgecolors='white', linewidths=0.4)
    ax.set_xlabel(feat_labels.get(fx, fx))
    ax.set_ylabel(feat_labels.get(fy, fy))
    ax.set_title(title)
    ax.legend(fontsize=9)

fig.suptitle('Gráficos de Dispersão — Relações entre Atributos por Variedade', fontsize=13)
plt.tight_layout()
plt.savefig('seeds_scatter.png', bbox_inches='tight')
plt.show()

**Interpretação dos Scatter Plots:**
- **Área × Perímetro:** clusters extremamente bem separados. Rosa (rosa) ocupa o canto superior direito; Canadian (verde) o inferior esquerdo; Kama (azul) no meio. Um classificador simples já distingue bem com apenas essas duas features.
- **Comp. Núcleo × Larg. Núcleo:** separação igualmente clara. Há leve sobreposição entre Kama e Canadian na região de comp. núcleo ~5,4–5,7, mas os clusters são distintos.
- **Assimetria × Compacidade:** mais sobreposição, especialmente entre Kama e Rosa. Confirma que compacidade e assimetria isoladas têm menor poder discriminativo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1, mask=mask,
    linewidths=0.5, ax=ax, annot_kws={'size': 10}
)
ax.set_title('Matriz de Correlação entre os Atributos')
plt.tight_layout()
plt.savefig('seeds_correlacao.png', bbox_inches='tight')
plt.show()

**Interpretação da Correlação:**
- **Área ↔ Perímetro (0.99):** correlação quase perfeita — ambos medem "tamanho" do grão. Em modelos lineares, isso pode causar multicolinearidade.
- **Área ↔ Comprimento do Núcleo (0.95)** e **Área ↔ Comprimento do Sulco (0.86):** alta colinearidade nas features de tamanho.
- **Compacidade ↔ Demais (negativa):** compacidade é inversamente correlacionada com tamanho — grãos maiores tendem a ser menos compactos, o que faz sentido pela fórmula $4\pi A / P^2$.
- **Assimetria** é a feature menos correlacionada com as demais — fornece informação independente ao modelo.

### 2.5 Tratamento de Valores Ausentes e Escalamento

In [ ]:
# ── 1. Verificação de nulos ───────────────────────────────────────────────────
nulos = df[FEATURES].isnull().sum()
print('Valores nulos por atributo:')
print(nulos)
print()

# ── 2. Preparação de X e y ───────────────────────────────────────────────────
X = df[FEATURES].values
y = df['variedade'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)   # Kama=0, Rosa=2, Canadian=1 (ordem alfabética)

print(f'Classes codificadas: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# ── 3. Split 70% treino / 30% teste (estratificado) ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=0.30,
    random_state=SEED,
    stratify=y_enc
)

print(f'\nTreino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X):.0%})')
print(f'Teste:  {X_test.shape[0]} amostras ({X_test.shape[0]/len(X):.0%})')
print()

# Verificação do balanceamento no split
for split_name, y_split in [('Treino', y_train), ('Teste', y_test)]:
    unique, counts = np.unique(y_split, return_counts=True)
    print(f'{split_name}: {dict(zip(le.classes_[unique], counts))}')

# ── 4. Normalização (StandardScaler) ─────────────────────────────────────────
# Ajustado APENAS no treino para evitar data leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('\nEscalamento aplicado com StandardScaler.')
print(f'Média pós-escala (treino): {X_train_sc.mean(axis=0).round(4)}')
print(f'Std  pós-escala (treino):  {X_train_sc.std(axis=0).round(4)}')

**Decisões de pré-processamento:**

| Etapa | Decisão | Justificativa |
|---|---|---|
| Valores ausentes | Nenhuma ação | Dataset limpo — zero nulos |
| Duplicatas | Nenhuma ação | Nenhuma duplicata encontrada |
| Split | 70/30 estratificado | Preserva proporção de classes em ambos os conjuntos |
| Escalamento | `StandardScaler` | Necessário para KNN e SVM (sensíveis à escala); aplicado apenas no treino |
| Modelos baseados em árvore | Dados não escalados | Random Forest não requer normalização |


<a id='3'></a>
## 3. Implementação e Comparação de Classificadores (CRISP-DM — Fase 4)

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = {}  # acumula métricas de todos os modelos

def avaliar_modelo(nome, clf, Xtr, ytr, Xte, yte):
    """Treina, avalia e exibe resultados de um classificador."""
    clf.fit(Xtr, ytr)
    y_pred = clf.predict(Xte)

    acc  = accuracy_score(yte, y_pred)
    prec = precision_score(yte, y_pred, average='macro', zero_division=0)
    rec  = recall_score(yte, y_pred, average='macro', zero_division=0)
    f1   = f1_score(yte, y_pred, average='macro', zero_division=0)
    cv_acc = cross_val_score(clf, Xtr, ytr, cv=cv_strategy, scoring='accuracy').mean()

    results[nome] = {
        'Acurácia': acc,
        'Precisão (macro)': prec,
        'Recall (macro)': rec,
        'F1-Score (macro)': f1,
        'CV Acurácia (5-fold)': cv_acc,
    }

    print(f'\n{'='*55}')
    print(f'  {nome}')
    print(f"{'='*55}")
    print(f'  Acurácia Teste:       {acc:.4f}')
    print(f'  Precisão Macro:       {prec:.4f}')
    print(f'  Recall Macro:         {rec:.4f}')
    print(f'  F1-Score Macro:       {f1:.4f}')
    print(f'  Acurácia CV 5-fold:   {cv_acc:.4f}')
    print()
    print(classification_report(yte, y_pred, target_names=le.classes_, digits=4))
    return clf, y_pred

print('Função auxiliar definida.')

### 3.1 K-Nearest Neighbors (KNN)

O KNN classifica uma amostra com base na votação majoritária de seus K vizinhos mais próximos no espaço de features. É intuitivo e não paramétrico, mas sensível à escala — por isso usa os dados normalizados.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_clf, knn_pred = avaliar_modelo('KNN (k=5)', knn, X_train_sc, y_train, X_test_sc, y_test)

### 3.2 Support Vector Machine (SVM)

O SVM com kernel RBF encontra o hiperplano de margem máxima no espaço transformado pelas features. Extremamente eficaz em datasets de dimensionalidade moderada e bem separáveis. Requer normalização.

In [ ]:
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED, probability=True)
svm_clf, svm_pred = avaliar_modelo('SVM (kernel RBF)', svm, X_train_sc, y_train, X_test_sc, y_test)

### 3.3 Random Forest

Ensemble de múltiplas árvores de decisão treinadas em subamostras aleatórias dos dados e das features. Reduz a variância (overfitting) em relação a uma única árvore. Não requer normalização.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_clf, rf_pred = avaliar_modelo('Random Forest (200 árvores)', rf, X_train, y_train, X_test, y_test)

### 3.4 Naive Bayes (Gaussiano)

Modelo probabilístico baseado no Teorema de Bayes com a suposição de independência condicional entre features. Extremamente rápido e serve como baseline probabilístico.

In [ ]:
nb = GaussianNB()
nb_clf, nb_pred = avaliar_modelo('Naive Bayes Gaussiano', nb, X_train_sc, y_train, X_test_sc, y_test)

### 3.5 Regressão Logística

Modelo linear que aprende fronteiras de decisão baseadas em combinações lineares das features. Interpretável, rápido e um excelente baseline para problemas bem separáveis.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=SEED, multi_class='auto', solver='lbfgs')
lr_clf, lr_pred = avaliar_modelo('Regressão Logística', lr, X_train_sc, y_train, X_test_sc, y_test)

### 3.6 Comparação dos Modelos

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('Acurácia', ascending=False)
print('=== Tabela Comparativa de Modelos (dados de teste) ===')
results_df.applymap('{:.4f}'.format)

In [ ]:
res_num = pd.DataFrame(results).T
cols_plot = ['Acurácia', 'Precisão (macro)', 'Recall (macro)', 'F1-Score (macro)', 'CV Acurácia (5-fold)']

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(res_num))
width = 0.16
bar_colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

for i, (col, color) in enumerate(zip(cols_plot, bar_colors)):
    bars = ax.bar(x + i * width - 2 * width, res_num[col], width,
                  label=col, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.004,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(res_num.index, rotation=15, ha='right', fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Comparação de Modelos — Métricas no Conjunto de Teste')
ax.legend(fontsize=8, loc='upper right')
ax.axhline(0.9, color='red', linestyle=':', linewidth=1, label='Meta 90%')
plt.tight_layout()
plt.savefig('seeds_comparacao_modelos.png', bbox_inches='tight')
plt.show()

In [ ]:
models_preds = [
    ('KNN (k=5)', knn_pred),
    ('SVM (kernel RBF)', svm_pred),
    ('Random Forest', rf_pred),
    ('Naive Bayes', nb_pred),
    ('Regressão Logística', lr_pred),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (name, pred) in enumerate(models_preds):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontsize=10)
    axes[i].tick_params(axis='x', labelsize=8)
    axes[i].tick_params(axis='y', labelsize=8)

axes[-1].set_visible(False)
fig.suptitle('Matrizes de Confusão — Todos os Modelos (Conjunto de Teste)', fontsize=13)
plt.tight_layout()
plt.savefig('seeds_matrizes_confusao.png', bbox_inches='tight')
plt.show()

### 3.7 Análise Comparativa

**Todos os modelos superam a meta de 90% de acurácia**, confirmando que o Seeds Dataset é bem separável.

| Modelo | Ponto Forte | Ponto Fraco |
|---|---|---|
| **KNN** | Simples, sem suposições distribicionais | Lento em inferência (calcula distâncias a todo treino); sensível à escala |
| **SVM (RBF)** | Robusto a outliers, ótimo em alta dimensão | Caixa-preta; hiperparâmetros C e γ críticos |
| **Random Forest** | Robusto, não precisa de escalamento, fornece importância de features | Lento em treino com muitas árvores |
| **Naive Bayes** | Extremamente rápido, interpretável | Suposição de independência viola a alta correlação encontrada nas features |
| **Reg. Logística** | Interpretável (coeficientes), rápido | Assume linearidade; pode sofrer com multicolinearidade (área × perímetro ≈ 0.99) |

**Padrão de erros nas matrizes de confusão:**
- A maioria dos erros ocorre entre **Kama e Canadian** — as duas variedades menores — especialmente em modelos mais simples (Naive Bayes, KNN).
- **Rosa** raramente é confundida com as demais (é o grão de maior porte).
- SVM e Random Forest tendem a errar menos que os demais.

<a id='4'></a>
## 4. Otimização de Hiperparâmetros — Grid Search (CRISP-DM — Fase 4, iteração 2)

In [ ]:
# Avaliamos se a otimização é necessária comparando os scores dos modelos iniciais
print('=== Necessidade de Otimização ===')
for nome, vals in results.items():
    acc = vals['Acurácia']
    cv  = vals['CV Acurácia (5-fold)']
    gap = acc - cv
    status = 'POTENCIAL DE MELHORA' if acc < 0.97 else 'Desempenho alto'
    print(f'{nome:35s} Acc={acc:.4f}  CV={cv:.4f}  Gap={gap:+.4f}  → {status}')

**Estratégia de otimização:**
Modelos com acurácia abaixo de 97% ou com gap CV/Teste significativo serão otimizados via **Grid Search com validação cruzada 5-fold** (estratificada). Otimizamos KNN, SVM e Random Forest — os três modelos centrais do requisito.

In [ ]:
print('=== Grid Search: KNN ===')

param_grid_knn = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15],
    'metric': ['euclidean', 'manhattan', 'minkowski'],
    'weights': ['uniform', 'distance'],
}

gs_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
gs_knn.fit(X_train_sc, y_train)

print(f'Melhores parâmetros: {gs_knn.best_params_}')
print(f'Melhor score CV:     {gs_knn.best_score_:.4f}')

knn_opt_pred = gs_knn.predict(X_test_sc)
knn_opt_acc  = accuracy_score(y_test, knn_opt_pred)
knn_opt_f1   = f1_score(y_test, knn_opt_pred, average='macro')
print(f'Acurácia Teste (otimizado): {knn_opt_acc:.4f}')
print(f'F1-Score Macro (otimizado): {knn_opt_f1:.4f}')
print(classification_report(y_test, knn_opt_pred, target_names=le.classes_, digits=4))

In [ ]:
print('=== Grid Search: SVM ===')

param_grid_svm = {
    'C': [0.1, 1, 5, 10, 50, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'kernel': ['rbf', 'poly', 'sigmoid'],
}

gs_svm = GridSearchCV(
    SVC(random_state=SEED, probability=True),
    param_grid_svm,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
gs_svm.fit(X_train_sc, y_train)

print(f'Melhores parâmetros: {gs_svm.best_params_}')
print(f'Melhor score CV:     {gs_svm.best_score_:.4f}')

svm_opt_pred = gs_svm.predict(X_test_sc)
svm_opt_acc  = accuracy_score(y_test, svm_opt_pred)
svm_opt_f1   = f1_score(y_test, svm_opt_pred, average='macro')
print(f'Acurácia Teste (otimizado): {svm_opt_acc:.4f}')
print(f'F1-Score Macro (otimizado): {svm_opt_f1:.4f}')
print(classification_report(y_test, svm_opt_pred, target_names=le.classes_, digits=4))

In [ ]:
print('=== Grid Search: Random Forest ===')

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
}

gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_grid_rf,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
gs_rf.fit(X_train, y_train)

print(f'Melhores parâmetros: {gs_rf.best_params_}')
print(f'Melhor score CV:     {gs_rf.best_score_:.4f}')

rf_opt_pred = gs_rf.predict(X_test)
rf_opt_acc  = accuracy_score(y_test, rf_opt_pred)
rf_opt_f1   = f1_score(y_test, rf_opt_pred, average='macro')
print(f'Acurácia Teste (otimizado): {rf_opt_acc:.4f}')
print(f'F1-Score Macro (otimizado): {rf_opt_f1:.4f}')
print(classification_report(y_test, rf_opt_pred, target_names=le.classes_, digits=4))

In [ ]:
opt_results = {
    'KNN — Baseline':       (results['KNN (k=5)']['Acurácia'],         results['KNN (k=5)']['F1-Score (macro)']),
    'KNN — Grid Search':    (knn_opt_acc, knn_opt_f1),
    'SVM — Baseline':       (results['SVM (kernel RBF)']['Acurácia'],   results['SVM (kernel RBF)']['F1-Score (macro)']),
    'SVM — Grid Search':    (svm_opt_acc, svm_opt_f1),
    'RF  — Baseline':       (results['Random Forest (200 árvores)']['Acurácia'],  results['Random Forest (200 árvores)']['F1-Score (macro)']),
    'RF  — Grid Search':    (rf_opt_acc, rf_opt_f1),
}

opt_df = pd.DataFrame.from_dict(
    opt_results, orient='index',
    columns=['Acurácia Teste', 'F1-Score Macro']
)
opt_df['Delta Acurácia'] = [
    0,
    knn_opt_acc - results['KNN (k=5)']['Acurácia'],
    0,
    svm_opt_acc - results['SVM (kernel RBF)']['Acurácia'],
    0,
    rf_opt_acc  - results['Random Forest (200 árvores)']['Acurácia'],
]

print('=== Comparação: Baseline vs. Grid Search Otimizado ===')
opt_df.applymap(lambda x: f'{x:+.4f}' if abs(x) < 0.5 else f'{x:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (model, base_key, opt_acc, opt_f1) in zip(axes, [
    ('KNN', 'KNN (k=5)',               knn_opt_acc, knn_opt_f1),
    ('SVM', 'SVM (kernel RBF)',         svm_opt_acc, svm_opt_f1),
    ('RF',  'Random Forest (200 árvores)', rf_opt_acc,  rf_opt_f1),
]):
    base_acc = results[base_key]['Acurácia']
    base_f1  = results[base_key]['F1-Score (macro)']

    x = np.array([0, 1])
    bar_width = 0.35

    ax.bar(x - bar_width/2, [base_acc, base_f1], bar_width,
           label='Baseline', color='#90CAF9', edgecolor='#1565C0', linewidth=1)
    ax.bar(x + bar_width/2, [opt_acc, opt_f1], bar_width,
           label='Grid Search', color='#A5D6A7', edgecolor='#1B5E20', linewidth=1)

    for val, xpos in zip([base_acc, base_f1, opt_acc, opt_f1],
                          [x[0]-bar_width/2, x[1]-bar_width/2,
                           x[0]+bar_width/2, x[1]+bar_width/2]):
        ax.text(xpos, val + 0.005, f'{val:.4f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(['Acurácia', 'F1-Score Macro'])
    ax.set_ylim(0.8, 1.1)
    ax.set_title(f'{model}: Baseline vs. Otimizado')
    ax.legend(fontsize=9)
    ax.axhline(0.9, color='red', linestyle=':', linewidth=1)

fig.suptitle('Impacto da Otimização de Hiperparâmetros por Modelo', fontsize=13)
plt.tight_layout()
plt.savefig('seeds_otimizacao.png', bbox_inches='tight')
plt.show()

### Análise da Otimização

**Houve melhora significativa?** Depende do modelo:

- **KNN:** o Grid Search tipicamente encontra $k < 5$ com peso `distance`, o que reduz o impacto de vizinhos distantes e melhora a separação nas fronteiras Kama/Canadian. Ganho esperado de 1–3%.

- **SVM:** o parâmetro `C` tem alto impacto. Um `C` maior (ex.: 10–50) permite margem menor mas classifica melhor os pontos de treino, reduzindo erros nos casos ambíguos entre Kama e Canadian. Ganho esperado de 0,5–2%.

- **Random Forest:** já com 200 árvores o baseline é robusto; ganhos com Grid Search são tipicamente marginais (< 0.5%). O parâmetro mais impactante é `max_features` — `sqrt` geralmente é ótimo para classificação.

**Conclusão sobre otimização:** todos os modelos já performavam bem acima da meta de 90%. O Grid Search confirma os hiperparâmetros ótimos e elimina dúvidas sobre configuração, mas os ganhos absolutos são pequenos — o que é esperado num dataset limpo e bem separável como o Seeds.

<a id='5'></a>
## 5. Interpretação dos Resultados (CRISP-DM — Fase 5)

In [ ]:
# Importância das features no Random Forest otimizado
best_rf = gs_rf.best_estimator_
importances = best_rf.feature_importances_
feat_names_pt = ['Área', 'Perímetro', 'Compacidade', 'Comp. Núcleo',
                 'Larg. Núcleo', 'Assimetria', 'Comp. Sulco']

feat_imp = pd.Series(importances, index=feat_names_pt).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
colors_fi = ['#d32f2f' if v == feat_imp.max() else '#42A5F5' for v in feat_imp.values]
feat_imp.plot.barh(ax=ax, color=colors_fi)
ax.set_title('Importância das Features — Random Forest Otimizado (Gini)')
ax.set_xlabel('Importância média (redução de impureza)')

for i, (v, name) in enumerate(zip(feat_imp.values, feat_imp.index)):
    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('seeds_importancia_features.png', bbox_inches='tight')
plt.show()

print('\nRanking de importância:')
print(feat_imp.sort_values(ascending=False).to_string())

In [ ]:
# Coeficientes da Regressão Logística (interpretabilidade linear)
coef_df = pd.DataFrame(
    lr_clf.coef_,
    columns=feat_names_pt,
    index=le.classes_
)
print('=== Coeficientes — Regressão Logística ===')
print('(valores positivos = feature aumenta probabilidade dessa classe)')
coef_df.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (classe, color) in zip(axes, zip(le.classes_, PALETTE)):
    coefs = coef_df.loc[classe].sort_values()
    bar_colors = ['#d32f2f' if c > 0 else '#1565C0' for c in coefs]
    coefs.plot.barh(ax=ax, color=bar_colors)
    ax.set_title(f'Coeficientes — Classe: {classe}', fontsize=10)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Coeficiente')

fig.suptitle('Regressão Logística — Coeficientes por Classe\n(vermelho = positivo, azul = negativo)',
             fontsize=12)
plt.tight_layout()
plt.savefig('seeds_coeficientes_lr.png', bbox_inches='tight')
plt.show()

In [ ]:
print('=== RESUMO FINAL DO PROJETO ===')
print()

all_models_summary = {}
for nome, vals in results.items():
    all_models_summary[nome] = vals['Acurácia']

# Inclui modelos otimizados
all_models_summary['KNN (Grid Search otimizado)']  = knn_opt_acc
all_models_summary['SVM (Grid Search otimizado)']  = svm_opt_acc
all_models_summary['RF  (Grid Search otimizado)']  = rf_opt_acc

best_name = max(all_models_summary, key=all_models_summary.get)
best_acc  = all_models_summary[best_name]

for nome, acc in sorted(all_models_summary.items(), key=lambda x: -x[1]):
    mark = '  ← MELHOR MODELO' if nome == best_name else ''
    print(f'  {nome:42s} Acurácia: {acc:.4f} ({acc:.1%}){mark}')

print()
print(f'Meta de negócio (≥90%): {"ATINGIDA" if best_acc >= 0.9 else "NÃO ATINGIDA"}')
print(f'Melhor modelo: {best_name} — {best_acc:.4f} ({best_acc:.1%})')

### 5.1 Interpretação Detalhada dos Resultados

#### Importância das Features (Random Forest)

A análise de importância por redução de impureza (Gini) revela:
1. **Comprimento do Núcleo** e **Área** são as features mais discriminativas — capturam diretamente o tamanho do grão, que difere significativamente entre Rosa (maior) e Canadian (menor).
2. **Comprimento do Sulco** também contribui fortemente, corroborado pela alta correlação com comprimento do núcleo.
3. **Compacidade** e **Assimetria** são as menos importantes individualmente, mas ainda contribuem para a distinção entre Kama e Canadian (as duas variedades de tamanho similar).

#### Coeficientes da Regressão Logística

- Para **Rosa**: coeficientes positivos em área e perímetro — confirma que grãos maiores são Rosa.
- Para **Canadian**: coeficientes negativos em área, perímetro e comprimentos — grãos menores identificam Canadian.
- Para **Kama**: perfil intermediário, distinguido principalmente por compacidade e assimetria.

#### Erros Remanescentes

Os erros persistentes em todos os modelos ocorrem entre **Kama e Canadian**. Essas duas variedades têm tamanhos mais próximos; os casos ambíguos correspondem a grãos de Kama levemente menores que a média ou de Canadian levemente maiores. Para reduzir esses erros seria necessário:
- Coletar mais amostras nos extremos de tamanho (Kama pequeno / Canadian grande).
- Adicionar features de textura da superfície do grão (não disponíveis neste dataset).

### 5.2 Conclusões Gerais

1. **Viabilidade da automação:** todos os modelos testados superam 90% de acurácia. Um sistema baseado em SVM ou Random Forest otimizados pode ser implantado em cooperativas agrícolas com alta confiabilidade.

2. **Modelo recomendado para produção:** **SVM com kernel RBF (otimizado)** — oferece a melhor combinação de acurácia, robustez a outliers e tempo de inferência rápido para batches pequenos (tamanho típico de cooperativas de pequeno porte).

3. **Dataset bem estruturado:** a limpeza dos dados e o balanceamento perfeito das classes simplificaram todo o pipeline de ML. Em dados reais de campo, etapas de imputação, normalização adicional e resampling seriam necessárias.

4. **Otimização teve impacto moderado:** o Grid Search confirmou configurações ótimas e trouxe ganhos marginais. Isso indica que os modelos padrão já eram adequados para o problema — investimento em coleta de dados adicionais seria mais impactante.

5. **Features de tamanho dominam:** área, perímetro e comprimento do núcleo são suficientes para uma triagem inicial de alta precisão. A compacidade e assimetria adicionam valor marginal, sendo mais relevantes para os casos ambíguos Kama/Canadian.

### 5.3 Próximos Passos (CRISP-DM — Fase 6: Implantação)

- **Prototipagem:** empacotar o modelo SVM otimizado com o `StandardScaler` em um `Pipeline` do scikit-learn e exportar via `joblib` para integração com o sistema da cooperativa.
- **API REST:** criar um endpoint simples (FastAPI ou Flask) que recebe os 7 atributos físicos e retorna a variedade classificada com probabilidade.
- **Interface de campo:** aplicativo mobile ou planilha Excel com macro para envio de dados ao classificador.
- **Monitoramento:** implementar um dashboard de acurácia contínua para detectar drift de distribuição (ex.: novos lotes com características fora do padrão histórico).
- **Expansão:** coletar dados de novas safras e regiões para tornar o modelo mais generalista.

In [ ]:
# Pipeline final recomendado para produção
from sklearn.pipeline import Pipeline
import joblib

best_svm_params = gs_svm.best_params_
pipeline_final = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(**best_svm_params, random_state=SEED, probability=True))
])

# Treino no dataset completo (todos os 210 registros)
pipeline_final.fit(X, y_enc)

# Exporta modelo
joblib.dump(pipeline_final, 'seeds_classifier_svm.joblib')
joblib.dump(le,             'seeds_label_encoder.joblib')

print('Pipeline SVM exportado: seeds_classifier_svm.joblib')
print('Label encoder exportado: seeds_label_encoder.joblib')
print()

# Demonstração de uso
amostra_nova = np.array([[15.26, 14.84, 0.871, 5.763, 3.312, 2.221, 5.220]])
predicao = pipeline_final.predict(amostra_nova)
prob      = pipeline_final.predict_proba(amostra_nova)

print(f'Amostra de teste: {amostra_nova[0]}')
print(f'Variedade predita: {le.inverse_transform(predicao)[0]}')
for i, (classe, p) in enumerate(zip(le.classes_, prob[0])):
    print(f'  P({classe}) = {p:.4f}')